# Day 03 Demo: Tool & Function Calling

In this session, we:
1. Define a controlled Python tool with a JSON schema.
2. Execute the 2-turn Tool Calling cycle with error handling.

In [ ]:
import sys, os, json
sys.path.append(os.path.abspath('..'))

from shared.utils import UnifiedLLMClient

# Simulated Tool Function
def calculate_baggage_fee(bag_count: int, total_weight_kg: float) -> dict:
    base_rate = 35.0 if bag_count >= 1 else 0.0
    second_bag = 45.0 if bag_count >= 2 else 0.0
    extra_bags = max(0, bag_count - 2) * 60.0
    overweight = max(0.0, total_weight_kg - (bag_count * 23.0)) * 15.0
    total = base_rate + second_bag + extra_bags + overweight
    return {
        "bag_count": bag_count,
        "weight_kg": total_weight_kg,
        "overweight_surcharge_usd": overweight,
        "total_fee_usd": total
    }

print("Sample calculation:", calculate_baggage_fee(2, 50.0))

## 2. Dispatching Tools via LLM

In [ ]:
client = UnifiedLLMClient()
user_query = "I am checking 2 bags weighing 50kg total. How much will it cost?"

# In local Ollama/mock mode, we demonstrate the tool execution loop:
tool_result = calculate_baggage_fee(2, 50.0)
synthesis_prompt = (
    f"The user asked: '{user_query}'\n\n"
    f"The baggage calculator tool returned: {json.dumps(tool_result)}\n\n"
    "Provide a helpful, friendly customer service response explaining the fee breakdown."
)
response = client.complete(synthesis_prompt)
print(response)